# Name That LLM: the final LLM runs, session by session

**Goal:** train the LLM member of the final ensemble (P7 Task 7). Notebook 11 chose its
settings: Qwen3-8B, full depth, 512 tokens, and 5,000 pool passages per fold. That lifted
the LLM alone from 0.373 to 0.444 on two proxy folds. But with the pool, two folds take 7
hours on the two T4s, so a full 5-fold run is more than one Kaggle session. The runs are
split into sessions. Each one runs the jobs listed in the **Jobs** cell, and each job
trains some folds of one seed, one fold per T4.

**This session asks one question: is 1 epoch enough?**
- Notebook 11 ran 2 epochs on seed-7 folds 0 and 1: 0.474 and 0.414, mean 0.444, in 420
  minutes.
- This session runs 1 epoch on the same folds, about 214 minutes.
- Within 0.005 of 0.444, 1 epoch wins (the cheaper setting, as in notebook 11). Every later
  run then costs half, and all of seed 42 fits in one session.

The blend, the report and the A/B gate run on the PC, from the saved fold predictions.

Part of [github.com/Ryqn0/name-that-llm](https://github.com/Ryqn0/name-that-llm), plan P7
(Task 7).

In [ ]:
# Pinned to an exact commit, so this run is reproducible even after main moves on.
# peft (LoRA) and bitsandbytes (the 4-bit weights) aren't in Kaggle's image.
%pip install -q peft bitsandbytes "name-that-llm[track] @ git+https://github.com/Ryqn0/name-that-llm@ec344d3a6a5633020cc9b52538f273544ffdfbec"

In [ ]:
import json
import os
import subprocess
import sys
import time
from contextlib import ExitStack
from pathlib import Path

import numpy as np
import pandas as pd
import peft
import torch
import transformers
import yaml

from name_that_llm.config import LLMConfig, load_config
from name_that_llm.cv import make_folds
from name_that_llm.data import LABELS, load_train
from name_that_llm.lm import find_model_dir
from name_that_llm.metrics import score
from name_that_llm.pool import build_pool, load_pool_sources

START = time.time()
# Kaggle's folders by default. The smoke test (tests/test_notebooks.py) runs this very code
# on a CPU, with these pointed at a tiny random Qwen, a fake pool and fake inputs.
INPUT = Path(os.environ.get("NTL_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("NTL_WORK", "/kaggle/working"))
CONFIG = os.environ.get("NTL_LLM_CONFIG", "llm")  # a config name, or a YAML path
ON_KAGGLE = "NTL_INPUT" not in os.environ
SESSION_MIN = 11 * 60  # Kaggle stops a session at 12 hours: keep an hour's margin

try:
    import bitsandbytes

    BNB = bitsandbytes.__version__
except ImportError:
    BNB = "not installed"
print(
    f"torch {torch.__version__} (CUDA {torch.version.cuda}) | transformers "
    f"{transformers.__version__} | peft {peft.__version__} | bitsandbytes {BNB}"
)
GPUS = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", GPUS)
if ON_KAGGLE:
    assert len(GPUS) == 2, "this notebook needs the T4 x2 accelerator"

train = load_train()
y = train["model"].to_numpy()
labels = np.asarray(LABELS)
CFG = load_config(CONFIG, model_cls=LLMConfig).model_copy(
    update={"base_model": str(find_model_dir("8b-base", INPUT))}
)
print(CFG)

## 1. The pool, rebuilt from the attached public dataset

The same code and safeguards as on the PC and in notebook 11 (`pool.build_pool`): open
licenses only, copies of our passages and conflicting labels removed, each passage matched
to the training prompt it's about (no fold trains on its own prompts' topics), and only the
5 sources that P7 Task 3 kept. Counts only are printed, never a passage.

In [ ]:
pool = build_pool(load_pool_sources("pool"), INPUT, train)
POOL_PATH = WORK / "pool.parquet"
pool.to_parquet(POOL_PATH)
print(f"pool: {len(pool):,} passages, {pool['prompt_id'].notna().mean():.1%} matched to a prompt")

## 2. Running one job: some folds of one seed, one fold per T4

Each fold runs as its own `python -m name_that_llm.llm` process on its own GPU. If one
fails, the other is stopped at once. Each fold's out-of-fold predictions are saved, and,
for the final seeds, its LoRA adapter too (`adapters/<job>/fold<k>/`), for the
submission.

In [ ]:
# Notebook 11 measured the pooled run: 420 minutes for 2 epochs on folds of about 7,160
# passages (our ~2,160 plus 5,000 from the pool), one fold per T4.
MIN_PER_PASSAGE_EPOCH, SCALE = 419.8 / (2 * 7160), [1.0]  # SCALE: learned from each job
results = []


def elapsed_minutes() -> float:
    return (time.time() - START) / 60


def on_gpu(gpu: int) -> dict:
    return {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONIOENCODING": "utf-8"}


def run_job(name: str, seed: int, fold_ids: tuple, overrides: dict, save: bool) -> dict | None:
    """Train folds `fold_ids` of `seed` (at most 2, one per T4); None if they wouldn't fit."""
    assert 1 <= len(fold_ids) <= 2, "one fold per GPU"
    cfg = CFG.model_copy(update={"seed": seed, **overrides})
    folds = make_folds(train, cfg.n_splits, seed).to_numpy()
    passages = max(int((folds != k).sum()) for k in fold_ids) + min(cfg.pool_size, len(pool))
    minutes = SCALE[0] * MIN_PER_PASSAGE_EPOCH * passages * cfg.epochs
    left = SESSION_MIN - elapsed_minutes()
    print(
        f"--> {name}: seed {seed}, folds {list(fold_ids)}, {cfg.epochs} epoch(s), "
        f"{passages:,} passages a fold: projected {minutes:.0f} min ({left:.0f} min left)",
        flush=True,
    )
    if minutes > left:
        print("    skipped: it wouldn't finish inside the session")
        return None
    folder = WORK / "runs" / name
    folder.mkdir(parents=True, exist_ok=True)
    (folder / "llm.yaml").write_text(yaml.safe_dump(cfg.model_dump()), encoding="utf-8")
    start, procs = time.time(), []
    with ExitStack() as logs:  # every log is closed, however the wait ends
        for gpu, k in enumerate(fold_ids):
            cmd = [sys.executable, "-m", "name_that_llm.llm", "--config", str(folder / "llm.yaml")]
            cmd += ["--seed", str(seed), "--folds", str(k), "--out", str(folder / f"fold{k}.npz")]
            if cfg.pool_size > 0:
                cmd += ["--pool", str(POOL_PATH)]
            if save:
                cmd += ["--save-dir", str(WORK / "adapters" / name)]
            log = logs.enter_context(open(folder / f"fold{k}.log", "w"))
            procs.append(
                subprocess.Popen(cmd, env=on_gpu(gpu), stdout=log, stderr=subprocess.STDOUT)
            )
        while None in (codes := [p.poll() for p in procs]):
            if any(code not in (None, 0) for code in codes):
                break  # one failed: stop the other below, rather than wait for it
            time.sleep(15)
        for p in procs:
            if p.poll() is None:
                p.kill()
                p.wait()
    failed = [k for k, p in zip(fold_ids, procs, strict=True) if p.returncode != 0]
    if failed:
        tail = (folder / f"fold{failed[0]}.log").read_text(encoding="utf-8", errors="replace")
        print(tail[-3000:])
        raise RuntimeError(f"{name}: fold {failed[0]} failed")
    scores = {}
    for k in fold_ids:
        with np.load(folder / f"fold{k}.npz") as npz:
            scores[k] = score(y[folds == k], labels[npz[f"fold{k}"].argmax(axis=1)])
    actual = (time.time() - start) / 60
    SCALE[0] *= actual / minutes  # the next projections learn from this job
    row = {"name": name, "seed": seed, "epochs": cfg.epochs, "pool_size": cfg.pool_size}
    row |= {f"fold{k}": s for k, s in scores.items()}
    row |= {"mean": float(np.mean(list(scores.values()))), "minutes": round(actual, 1)}
    results.append(row)
    pd.DataFrame(results).to_csv(WORK / "results.csv", index=False)  # saved after every job
    print(
        f"    {name}: folds "
        + ", ".join(f"{s:.4f}" for s in scores.values())
        + f" | mean {row['mean']:.4f} | {actual:.0f} min",
        flush=True,
    )
    return row

## 3. Jobs

Each job is (name, seed, folds, overrides of `configs/llm.yaml`, save the adapters?). This
session: the pooled LLM at 1 epoch, on the seed-7 folds that notebook 11 ran at 2 epochs.
Its adapters aren't kept: these folds aren't the final seed's.

In [ ]:
JOBS = [("ep1-seed7", 7, (0, 1), {"epochs": 1}, False)]
for job in JOBS:
    run_job(*job)

## 4. The verdict: 1 epoch or 2

The 2-epoch reference is notebook 11's `pool5000` row (attached). Within 0.005 of its
mean, 1 epoch wins. Fine-tuning is noisy (P5 saw 0.014 between identical runs on two
folds), so a near tie means "1 epoch is enough", not "1 epoch is better".

In [ ]:
explore = [p for p in INPUT.rglob("results.csv") if "llm-explore" in str(p)]
assert len(explore) == 1, f"expected notebook 11's results, found {explore}"
ref = pd.read_csv(explore[0]).set_index("name").loc["pool5000"]
reference = {k: float(ref[k]) for k in ("fold0", "fold1", "mean")}
print(pd.DataFrame(results).to_string(index=False))
ep1 = next((r for r in results if r["name"] == "ep1-seed7"), None)
summary = {"reference": reference, "ep1": ep1, "chosen_epochs": 2}
if ep1 is not None:
    delta = ep1["mean"] - reference["mean"]
    summary |= {"delta": delta, "chosen_epochs": 1 if delta >= -0.005 else 2}
    print(f"1 epoch: {ep1['mean']:.4f} | 2 epochs (notebook 11): {reference['mean']:.4f}")
    print(f"delta {delta:+.4f} -> {summary['chosen_epochs']} epoch(s)")
(WORK / "results.json").write_text(json.dumps(summary, indent=1), encoding="utf-8")

## Findings

_Filled in after the run._